# 🧠 How does a model see the world?

<a href="https://colab.research.google.com/github/racousin/scai_team/blob/main/how_models_see_the_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Every language model turns a text into a list of numbers — an **embedding**. That vector is, quite literally, how the model *sees* the text: two texts the model considers similar end up with nearby vectors.

In this notebook you will:

1. Take short bios — **the SCAI team**, cartoon characters, politicians.
2. Ask **two different small models** from 🤗 Hugging Face to read them.
3. Project each model's vectors down to 2D and **look at the world through each model's eyes**.
4. Then feed them **political opinions** — and catch each model's bias red-handed.

▶️ **Runtime ▸ Run all**, then scroll down to the plots (≈1 min on the free CPU runtime). After that, come back and start editing the texts.

In [ ]:
!pip install -q sentence-transformers
!wget -q -O embedding_utils.py https://raw.githubusercontent.com/racousin/scai_team/main/embedding_utils.py
from embedding_utils import *

## 1. The texts

This is the only "data": one short bio per character, tagged with a `group`. Nothing else.

The team bios are built from the roles on [scai.sorbonne-universite.fr/management](https://scai.sorbonne-universite.fr/management) — on purpose, they say little more than a job title. Keep that in mind for later.

In [ ]:
ITEMS = [
    # --- the SCAI team (roles from scai.sorbonne-universite.fr/management) ---
    {"name": "Gérard Biau", "group": "SCAI team",
     "text": "Director of SCAI, the Sorbonne Center for Artificial Intelligence in Paris."},
    {"name": "Xavier Fresquet", "group": "SCAI team",
     "text": "Deputy director of SCAI, the Sorbonne Center for Artificial Intelligence in Paris."},
    {"name": "Nora Roger", "group": "SCAI team",
     "text": "Administrative and financial manager of SCAI at Sorbonne University."},
    {"name": "Dimitris Alchatzidis", "group": "SCAI team",
     "text": "Project manager at SCAI, running artificial intelligence projects at Sorbonne University."},
    {"name": "Clotilde Chevet", "group": "SCAI team",
     "text": "Project manager at SCAI, coordinating interdisciplinary AI projects in Paris."},
    {"name": "Vivien Conti", "group": "SCAI team",
     "text": "Data scientist at SCAI, the Sorbonne Center for Artificial Intelligence."},
    {"name": "Omar El Dakkak", "group": "SCAI team",
     "text": "Associate professor in mathematics at Sorbonne University, member of the SCAI team."},
    {"name": "Baptiste Gregorutti", "group": "SCAI team",
     "text": "Research engineer at SCAI, the Sorbonne Center for Artificial Intelligence in Paris."},
    {"name": "Étienne Guével", "group": "SCAI team",
     "text": "Data scientist at SCAI, working on machine learning at Sorbonne University."},
    {"name": "Linghwei Loubaresse", "group": "SCAI team",
     "text": "Project manager at SCAI, the Sorbonne Center for Artificial Intelligence."},
    {"name": "Rakhee Patel", "group": "SCAI team",
     "text": "Strategic partnership manager at SCAI, connecting the center with its partners."},
    {"name": "Julie Perrin", "group": "SCAI team",
     "text": "Communication officer at SCAI, telling the story of AI research at Sorbonne University."},
    {"name": "Alain Rabaute", "group": "SCAI team",
     "text": "Manager of the SOUND.AI programme at SCAI, Sorbonne University."},
    {"name": "Julien Roudil", "group": "SCAI team",
     "text": "Operational manager of the Campus for AI professions and qualifications at Sorbonne University."},
    {"name": "Marco Salazar", "group": "SCAI team",
     "text": "Education project manager at SCAI, the Sorbonne Center for Artificial Intelligence."},
    # 👇 ADD YOURSELF — uncomment, put your name and a 1–2 sentence bio, re-run everything!
    # {"name": "YOUR NAME", "group": "SCAI team",
    #  "text": "I am ..."},

    # --- cartoons -----------------------------------------------------------
    {"name": "Homer Simpson", "group": "cartoon",
     "text": "Lazy but lovable father from Springfield who works at a nuclear power plant, loves donuts and beer, and always says d'oh."},
    {"name": "Mickey Mouse", "group": "cartoon",
     "text": "Cheerful cartoon mouse with big round ears and red shorts who whistles, laughs, and stars in Disney adventures."},
    {"name": "Naruto", "group": "cartoon",
     "text": "Hyperactive young ninja from the Hidden Leaf Village who eats ramen, never gives up, and dreams of becoming Hokage."},
    {"name": "Elsa", "group": "cartoon",
     "text": "Ice queen from Arendelle with magical powers to freeze everything around her, who sings about letting it go."},
    {"name": "SpongeBob", "group": "cartoon",
     "text": "Optimistic yellow sea sponge who lives in a pineapple under the sea and flips burgers at the Krusty Krab."},

    # --- politicians ---------------------------------------------------------
    {"name": "Macron (EN)", "group": "politician",
     "text": "President of France, former investment banker, who leads the country from the Élysée Palace in Paris."},
    {"name": "Macron (FR)", "group": "politician",
     "text": "Président de la République française, ancien banquier d'affaires, qui dirige le pays depuis le palais de l'Élysée à Paris."},
    {"name": "Barack Obama", "group": "politician",
     "text": "Former President of the United States, known for his eloquent speeches, healthcare reform, and a Nobel Peace Prize."},
    {"name": "Angela Merkel", "group": "politician",
     "text": "Former Chancellor of Germany, a trained physicist who led Europe's largest economy for sixteen years."},
    {"name": "Nelson Mandela", "group": "politician",
     "text": "South African anti-apartheid revolutionary who spent 27 years in prison and became his country's first Black president."},
]

## 2. The models

Two **small** models from the [Hugging Face hub](https://huggingface.co/models?library=sentence-transformers). Same job, different upbringing:

| | parameters | trained on |
|---|---|---|
| [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | 22M | **English only** |
| [`paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) | 118M | **50+ languages** |

Any other model name from the hub works here too — that's exercise 5.

In [ ]:
MODEL_A = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_B = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## 3. Two world views

Each model reads the same texts. Each dot below is one bio, placed on each model's own 2D map. **Hover a dot** to see the name and the full bio — in crowded spots only the outer names are printed, so if you can't find yours, it's hiding inside the team blob (escape it in exercise 1 and your name reappears).

In [ ]:
compare_models(ITEMS, MODEL_A, MODEL_B)

### What are you looking at?

- Each model turns a bio into a vector of 384 numbers; we squash those down to 2D. **The axes mean nothing — only distances do.** Close dots = "these texts look similar *to this model*".
- Do the three groups form clusters? In both models? Who ended up nearest to the SCAI team, and why?
- The team forms a *very* tight blob — and some teammates are almost the same point. The model has never met anyone: it only reads the words, and bios that all say *"… manager at SCAI, Sorbonne …"* are, to it, nearly the same text. **You are where your words are.**
- 🔍 **Find the two Macrons.** Same bio, two languages. The multilingual model (right) sees them as *the same text* (similarity ≈ 0.95 — the dots overlap). The English-only model (left) only vaguely relates them (≈ 0.6), and mostly because of the shared names *Élysée* and *Paris* — exercise 3 pushes on exactly this weakness.
- Same texts, different training data → **a genuinely different view of the world**. Neither map is "the truth"; each is what the model learned to pay attention to.

*(Tip: `compare_models(ITEMS, MODEL_A, MODEL_B, method="tsne")` gives an alternative projection that exaggerates clusters.)*

## 4. Who is closest to whom?

The 2D map is a lossy summary. The matrix below is the real thing: the similarity the model assigns to **every pair** of texts (1.0 = identical view). Swap in `MODEL_A` and see which pairs change the most.

In [ ]:
similarity_heatmap(ITEMS, MODEL_B)

## 5. Politics 🗳️ — when models stop agreeing

So far our two models mostly agreed: bios cluster by *topic*, and topic is the only thing that varies much between them. Time to make the models' **bias** flagrant — feed them texts where "similar" is genuinely ambiguous: **political opinions**.

8 statements, built as a grid: **4 topics × 2 camps** (left / right), and — independently of the camp — half are written in a **hopeful 😊 tone**, half in an **angry 😠 tone**.

Before running: which statements do *you* consider similar? Same topic? Same camp? Same mood? Each model has already made its choice — and they don't agree with each other (or with you).

In [ ]:
OPINIONS = [
    {"name": "Climate — left 😊", "group": "left-leaning",
     "text": "Investing in solar and wind will create millions of good jobs and give our children a cleaner future."},
    {"name": "Climate — right 😠", "group": "right-leaning",
     "text": "Green mandates are strangling our industries and driving up energy bills for working families."},
    {"name": "Immigration — right 😊", "group": "right-leaning",
     "text": "With secure borders and merit-based selection, we can build an immigration system that truly serves our nation."},
    {"name": "Immigration — left 😠", "group": "left-leaning",
     "text": "Deporting families who came here seeking safety is a cruel betrayal of everything this country stands for."},
    {"name": "Taxes — left 😠", "group": "left-leaning",
     "text": "Billionaires dodge taxes while nurses and teachers struggle to pay rent — this rigged system must end."},
    {"name": "Taxes — right 😊", "group": "right-leaning",
     "text": "Lower taxes let families keep more of what they earn and give small businesses room to grow and hire."},
    {"name": "Health — left 😊", "group": "left-leaning",
     "text": "Universal public healthcare would free every family from the fear of medical bills and let doctors focus on patients."},
    {"name": "Health — right 😠", "group": "right-leaning",
     "text": "Government-run healthcare means endless waiting lists and bureaucrats deciding who gets treated."},
]

# A model trained on 124M tweets to judge sentiment — NOT built for embeddings.
# We mean-pool its internals to ask it "how do you see this text?" (hence the warning below).
MODEL_C = "cardiffnlp/twitter-roberta-base-sentiment-latest"

compare_models(OPINIONS, MODEL_A, MODEL_C)

### Two models, two ideologies of "similar"

- **Left map (MiniLM)** — politics is organised by **topic**. Each subject forms a pair, and the two *opposing* camps nearly touch: to this model, *"tax the rich"* and *"cut taxes"* are almost the same sentence (similarity ≈ 0.44 — same words, same subject). **It literally cannot see the disagreement.**
- **Right map (the sentiment model)** — topic and camp both vanish. Everything lines up by **emotional tone**: 😊 on one side, 😠 on the other. To this model, an angry left text about *taxes* and an angry right text about *healthcare* are near-duplicates (≈ 0.91!), while a hopeful and an angry text from the *same camp* are far apart. It spent its training judging the mood of tweets — so mood is all it sees.
- And notice what **neither** map does: separate blue from green. The political axis you expected simply **does not exist** for these models.

This is model bias made visible: *what a model was trained for decides what it thinks "similar" means.* A recommender built on the left model would show you your opponents' posts about your favourite topic; one built on the right model would feed angry people more anger — from either camp. Same data, same math, different world views.

Also check `similarity_heatmap(OPINIONS, MODEL_C)` — and try `MODEL_B` on `OPINIONS`: which lens does the multilingual model use?

## 6. Your turn 🔬

Edit the cells above and re-run (the models stay cached, so re-runs are fast):

1. **Find yourself, then free yourself.** Your bio is just your job title, so you sit in the team blob — possibly on top of a colleague. Rewrite it with what you actually do and love (research topics, hobbies, an obsession) and watch yourself move. Not on the team? Use the `👇 ADD YOURSELF` slot.
2. **Rewrite your bio as a cartoon character** ("…lives in a pineapple…") and watch yourself migrate across the map. What does that tell you about what the model reads: your name, or your words?
3. **Translate a bio that shares no names between languages** — e.g. add SpongeBob in French: *"Éponge de mer jaune et optimiste qui vit dans un ananas sous la mer et retourne des burgers."* For the English-only model the two SpongeBobs become near-strangers (similarity ≈ 0.3 — check the heatmap!), while the multilingual one keeps them together (≈ 0.7). Which model would you trust to search French customer feedback?
4. **Fool the map**: write one sentence that lands exactly *between* the politicians and the cartoons (a president of Springfield?).
5. **Swap a model**: put any name from [huggingface.co/models](https://huggingface.co/models?library=sentence-transformers) into `MODEL_A` — e.g. `"intfloat/multilingual-e5-small"` — and see how *its* world differs.
6. **Break the politics experiment**: add a *calm, purely factual* sentence about taxes ("The standard VAT rate in France is 20%.") to `OPINIONS`. Where does each model put it — with the tax opinions, or somewhere else entirely?
7. **The hard one**: neither model separates left from right. What would it take to build one that does? What texts would you train it on — and would *that* model's map be more "true", or just biased your way?

### What you just learned

An embedding is a model's world view: texts become points, meaning becomes distance. Different training data or objective → different geometry — MiniLM thinks opposite opinions on one topic are twins; a sentiment model thinks angry texts from rival camps are twins. **That is what bias is in an embedding model**: not an opinion the model states, but the choice of which differences it can see at all. This exact mechanism powers semantic search, recommendation, clustering and RAG — there, too, the model's world view (not yours) decides what counts as "similar".